In [25]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, Generator
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage

In [26]:
class SubGraph(TypedDict):
    input_text : str
    translated_text : str

In [27]:
sub_graph_llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

In [48]:
def translate_text(state : SubGraph):
    prompt = f"""Translate the following text to Bangla, 
    keep it natural and clear. Do not add extra content . 
    Text : {state['input_text']}""".strip()
    translated_text = sub_graph_llm.invoke(prompt).content
    return {"translated_text": translated_text }

In [56]:
sub_graph_builder = StateGraph(SubGraph)

sub_graph_builder.add_node('translate_text', translate_text)
sub_graph_builder.add_edge(START, 'translate_text')
sub_graph_builder.add_edge('translate_text', END)

subgraph = sub_graph_builder.compile()

In [57]:
class ParentState(TypedDict):
    question : str
    answer_eng : str
    answer_hin : str

In [58]:
parent_graph_llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

In [59]:
def generate_answer(state: ParentState):
    answer = parent_graph_llm.invoke(f"you are a helpful assistant. Answer clearly.\n\nQuestion : {state['question']}").content
    return {
        'answer_eng' : answer
    }
def translate_answer(state: ParentState):
    result = subgraph.invoke({'input_text': state['answer_eng']})
    return {'answer_hin' : result['translated_text']}

In [62]:
parent_builder = StateGraph(ParentState)
parent_builder.add_node('answer', generate_answer) 
parent_builder.add_node('translate', translate_answer)

parent_builder.add_edge(START, 'answer')
parent_builder.add_edge('answer', 'translate')
parent_builder.add_edge('translate', END)

ouput = parent_builder.compile()

In [64]:
res = ouput.invoke({
    'question': 'What is organic chemestry'
})


In [65]:
res

{'question': 'What is organic chemestry',
 'answer_eng': "Organic chemistry is a branch of chemistry that focuses on the study of carbon-containing compounds, which form the basis of life and are essential for many aspects of modern technology and industry. Here are some key points about organic chemistry:\n\n1. **Scope**: Organic chemistry encompasses the structure, properties, composition, reactions, and preparation of organic compounds.\n\n2. **Carbon Compounds**: It primarily deals with carbon-based molecules, which can form a vast array of structures due to carbon's ability to bond in multiple ways (single, double, or triple bonds) and its capacity to form long chains and rings.\n\n3. **Applications**: Organic chemistry has numerous practical applications, including the development of pharmaceuticals, plastics, fuels, and many other materials used in everyday life.\n\n4. **Subfields**: It includes subdisciplines such as synthetic organic chemistry (which focuses on creating new co